## Setup
Imports et initialisation du client Mistral.
On a besoin de : mistralai (API), numpy (calculs vectoriels), requests + beautifulsoup (scraping web).

In [10]:
from mistralai import Mistral
from dotenv import load_dotenv
import os
import numpy as np
import requests
from bs4 import BeautifulSoup

load_dotenv()
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
print("✅ Client Mistral initialisé")

✅ Client Mistral initialisé


## 1. Scraping
On récupère automatiquement le texte de 6 pages de docs.mistral.ai.
Pourquoi scraper ? Pour avoir une vraie source externe plutôt que des phrases codées en dur.
Résultat : ~43 000 caractères de documentation Mistral.

In [11]:
import requests
from bs4 import BeautifulSoup

def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Extraire uniquement le contenu principal (pas la nav, pas le footer)
    main = soup.find("main") or soup.find("article") or soup.body
    
    # Supprimer les balises de navigation
    for tag in main.find_all(["nav", "footer", "script", "style"]):
        tag.decompose()
    
    text = main.get_text(separator="\n", strip=True)
    # Nettoyer les lignes vides multiples
    lines = [l for l in text.splitlines() if l.strip()]
    return "\n".join(lines)

# Test sur la page platform overview
text = scrape_page("https://docs.mistral.ai/getting-started/platform-overview")
print(text[:500])

Platform overview
Mistral AI has three products. Pick the one that matches what you want to do.
Vibe
: the unified agent for productivity and coding. Chat with it on the web or mobile, or run it in your terminal and editor.
Studio
: the developer console and Mistral API. Generate keys, prototype in the Playground, build durable workflows, and call text, audio, and OCR models from a single SDK.
Admin
: the control plane for organization setup, billing, SSO, workspaces, and access policies.
Tip
Fo


In [12]:
# Pages clés de la doc Mistral
urls = {
    "platform_overview": "https://docs.mistral.ai/getting-started/platform-overview",
    "models_overview": "https://docs.mistral.ai/models/overview",
    "embeddings": "https://docs.mistral.ai/capabilities/embeddings",
    "function_calling": "https://docs.mistral.ai/capabilities/function_calling",
    "rag": "https://docs.mistral.ai/getting-started/quickstarts/developer/rag-document-search",
    "agents": "https://docs.mistral.ai/getting-started/quickstarts/developer/build-an-agent",
}

# Scraper toutes les pages
pages = {}
for name, url in urls.items():
    print(f"Scraping {name}...")
    pages[name] = scrape_page(url)

print(f"\n✅ {len(pages)} pages scrapées")
for name, text in pages.items():
    print(f"  {name} : {len(text)} caractères")

Scraping platform_overview...
Scraping models_overview...
Scraping embeddings...
Scraping function_calling...
Scraping rag...
Scraping agents...

✅ 6 pages scrapées
  platform_overview : 2235 caractères
  models_overview : 6507 caractères
  embeddings : 1250 caractères
  function_calling : 24616 caractères
  rag : 3753 caractères
  agents : 5003 caractères


## 2. Chunking
On découpe les 43 000 caractères en 99 morceaux de 500 caractères.
Pourquoi découper ? Un texte trop long dilue le sens — un chunk de 500 chars (~3-4 phrases) est la bonne granularité pour la recherche sémantique.
Pourquoi l'overlap de 50 chars ? Pour ne pas perdre les informations à cheval sur deux chunks.

In [13]:
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Découpe un texte en chunks de chunk_size caractères
    avec overlap caractères de chevauchement entre chunks.
    
    Pourquoi l'overlap ?
    Sans overlap : une info à cheval sur 2 chunks peut être perdue.
    Avec overlap : chaque chunk partage 50 caractères avec le suivant
                   → on ne rate pas les infos aux frontières.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Chunker toutes les pages + garder la source
all_chunks = []
for name, text in pages.items():
    chunks = chunk_text(text, chunk_size=500, overlap=50)
    for chunk in chunks:
        all_chunks.append({"source": name, "text": chunk})

print(f"Total chunks : {len(all_chunks)}")
print(f"\nExemple chunk :")
print(all_chunks[10]["source"], "→", all_chunks[10]["text"][:200])

Total chunks : 99

Exemple chunk :
models_overview → es.
v
26.02
Codestral
Premier
Our cutting-edge language model for code completion released end of July 2025.
v
25.08
Codestral Embed
Premier
Our state-of-the-art semantic for extracting representation


## 3. Embeddings
On transforme les 99 chunks en vecteurs de 1024 dimensions via mistral-embed.
Chaque vecteur représente le sens du chunk. Deux chunks similaires → vecteurs proches.
C'est notre base vectorielle — l'index dans lequel on va chercher.

In [14]:
# Embedder tous les chunks en une seule requête API
print("Embedding 99 chunks...")

texts = [c["text"] for c in all_chunks]

response = client.embeddings.create(
    model="mistral-embed",
    inputs=texts
)

chunk_embeddings = np.array([item.embedding for item in response.data])
print(f"Shape : {chunk_embeddings.shape}")  # attendu : (99, 1024)

Embedding 99 chunks...
Shape : (99, 1024)


## 4. Retrieval
On mesure la similarité cosinus entre la question et chaque chunk.
Pourquoi cosinus ? Il mesure la direction des vecteurs, pas leur magnitude — un texte long n'a pas un score artificiellement plus élevé.
On retourne les top-3 chunks les plus pertinents avec leur source.

In [15]:
def retrieve(question, top_k=3):
    # Embed la question
    q_response = client.embeddings.create(model="mistral-embed", inputs=[question])
    q_emb = np.array(q_response.data[0].embedding)
    
    # Similarité cosinus
    scores = chunk_embeddings @ q_emb
    scores = scores / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb))
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for i in top_indices:
        results.append({
            "source": all_chunks[i]["source"],
            "text": all_chunks[i]["text"],
            "score": scores[i]
        })
    return results

# Test
question = "C'est quoi les modèles disponibles chez Mistral ?"
results = retrieve(question)
for r in results:
    print(f"[{r['score']:.3f}] {r['source']}")
    print(f"  {r['text'][:100]}\n")

[0.781] models_overview
  
12/2/2024
6/16/2025
Codestral
Mistral 7B
↗
0.3
open-mistral-7b
11/30/2024
3/30/2025
Ministral 3 8B


[0.777] models_overview
  al Medium 3.5
Open
Our frontier-class multimodal model optimized for agentic and coding use cases.
v

[0.776] models_overview
  els
Copy section link
Other Models
Other Models
Other supported models available.
Mistral Embed
Prem



## 5. RAG complet
On assemble retrieval + génération.
Le system prompt force le LLM à répondre UNIQUEMENT depuis le contexte — pas d'hallucination.
Chaque réponse cite sa source pour la traçabilité.

In [16]:
def rag(question, top_k=3):
    # 1. Retrieval
    results = retrieve(question, top_k)
    
    # 2. Contexte avec sources
    context = ""
    for r in results:
        context += f"[Source: {r['source']}]\n{r['text']}\n\n"
    
    # 3. Génération
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[
            {"role": "system", "content": "Tu réponds UNIQUEMENT en te basant sur le contexte fourni. Cite toujours la source entre crochets. Si la réponse n'est pas dans le contexte, dis-le."},
            {"role": "user", "content": f"Contexte :\n{context}\nQuestion : {question}"}
        ]
    )
    return response.choices[0].message.content

# Tests
print(rag("Quels sont les modèles disponibles chez Mistral ?"))
print("\n---\n")
print(rag("Comment fonctionne le function calling ?"))
print("\n---\n")
print(rag("C'est quoi la différence entre Vibe et Studio ?"))

Voici la liste des modèles disponibles chez Mistral basée sur le contexte fourni :

### Modèles principaux :
- **Codestral** [Source: models_overview]
- **Mistral 7B** [Source: models_overview]
- **Ministral 3 8B** [Source: models_overview]
- **Mixtral 8x22B** [Source: models_overview]
- **Mistral Small 4** [Source: models_overview]
- **Mistral Small 1.0** [Source: models_overview]
- **Mistral Small 2.0** [Source: models_overview]
- **Mistral Large 1.0** [Source: models_overview]
- **Mistral Large 3** [Source: models_overview]
- **Mistral Next** [Source: models_overview]
- **Mistral Medium 1.0** [Source: models_overview]
- **Mistral Medium 3.5** [Source: models_overview]
- **Mixtral 8x7B** [Source: models_overview]
- **Pixtral Large** [Source: models_overview]
- **Mistral Moderation** [Source: models_overview]
- **Mistral Moderation 2** [Source: models_overview]
- **Ministral 3B** [Source: models_overview]
- **Ministral 3 3B** [Source: models_overview]
- **Ministral 8B** [Source: model